# 05e - HayFlow-Hines closed-form segment-capacity probe

This notebook determines whether the 05b frozen features need explicit segment identity and how much low-rank segment-conditioned capacity is required. It uses deterministic float64 linear algebra, never runs full training, and never authorizes it.

## 1. Coherent checkout and GPU runtime

In [ ]:
import os, subprocess, sys
from pathlib import Path
WORKSPACE = Path('/kaggle/working/hayflow_workspace')
ELM_REPO = WORKSPACE / 'elmneuron'
if not ELM_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Zagred47/giada.git', str(ELM_REPO)], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'pandas', 'pyarrow', 'pyyaml', 'matplotlib'], check=True)
sys.path.insert(0, str(ELM_REPO))
REVISION = subprocess.check_output(['git', '-C', str(ELM_REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Revision:', REVISION)

In [ ]:
import h5py, json, numpy as np, pandas as pd, pyarrow, torch, yaml
assert torch.cuda.is_available(), 'Attiva una GPU Kaggle: serve soltanto per estrarre le feature congelate H2.'
print({'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0), 'linear_algebra': 'NumPy float64 closed form'})

## 2. Inputs
Sono richiesti il composite targeted, il checkpoint 05b e gli artefatti esatti 05c e 05d. ZIP originali e directory estratte da Kaggle sono entrambi accettati e verificati tramite SHA-256.

In [ ]:
import shutil, zipfile
INPUT_ROOT = Path('/kaggle/input')
def extract_zip_safely(source, destination):
    source, destination = Path(source), Path(destination)
    marker = destination / '.source_size'
    stamp = str(source.stat().st_size)
    if marker.is_file() and marker.read_text().strip() == stamp: return destination
    if destination.exists(): shutil.rmtree(destination)
    destination.mkdir(parents=True)
    root = destination.resolve()
    with zipfile.ZipFile(source) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            assert target == root or root in target.parents, member.filename
        archive.extractall(destination)
    marker.write_text(stamp)
    return destination

topup_override = os.environ.get('HAYFLOW_TOPUP_V3')
topup_candidates = [Path(topup_override).expanduser()] if topup_override else []
topup_candidates.extend(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))
topup_candidates.extend(path.parent for path in INPUT_ROOT.rglob('composite_dataset_manifest.json'))
TOPUP_SOURCE = next((p.resolve() for p in topup_candidates if p.exists()), None)
assert TOPUP_SOURCE is not None, 'Top-up BAP v3 non trovato.'
TOPUP_ROOT = extract_zip_safely(TOPUP_SOURCE, '/kaggle/working/hayflow05e_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates = list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'))
assert len(manifest_candidates) == 1, manifest_candidates
COMPOSITE_MANIFEST = manifest_candidates[0]

base_override = os.environ.get('HAYFLOW_BASE_DATASET')
base_candidates = [Path(base_override).expanduser()] if base_override else []
base_candidates.extend(path.parent for path in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(path).lower() and 'topup' not in str(path).lower())
base_candidates.extend(path for path in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(path).lower())
BASE_SOURCE = next((p.resolve() for p in base_candidates if p.exists()), None)
assert BASE_SOURCE is not None, 'Dataset base targeted v1.1 non trovato.'

checkpoint_override = os.environ.get('HAYFLOW_05B_ARTIFACT')
checkpoint_candidates = [Path(checkpoint_override).expanduser()] if checkpoint_override else []
checkpoint_candidates.extend(INPUT_ROOT.rglob('hayflow_hines_canary_v2.zip'))
checkpoint_candidates.extend(path.parent.parent for path in INPUT_ROOT.rglob('canary_models.pt') if path.parent.name == 'checkpoints' and 'hayflow' in str(path).lower())
CHECKPOINT_05B_SOURCE = next((p.resolve() for p in checkpoint_candidates if p.exists()), None)
assert CHECKPOINT_05B_SOURCE is not None, 'Artefatto 05b non trovato come ZIP o cartella Kaggle estratta.'

causal_override = os.environ.get('HAYFLOW_05C_ARTIFACT')
causal_candidates = [Path(causal_override).expanduser()] if causal_override else []
causal_candidates.extend(INPUT_ROOT.rglob('hayflow_hines_causal_isolation.zip'))
causal_candidates.extend(path.parent for path in INPUT_ROOT.rglob('final_report.json') if (path.parent / 'checkpoint_forensics.json').is_file() and (path.parent / 'progressive_isolation_report.json').is_file())
ARTIFACT_05C_SOURCE = next((p.resolve() for p in causal_candidates if p.exists()), None)
assert ARTIFACT_05C_SOURCE is not None, 'Artefatto 05c non trovato.'

conditioning_override = os.environ.get('HAYFLOW_05D_ARTIFACT')
conditioning_candidates = [Path(conditioning_override).expanduser()] if conditioning_override else []
conditioning_candidates.extend(INPUT_ROOT.rglob('hayflow_hines_residual_conditioning.zip'))
conditioning_candidates.extend(path.parent for path in INPUT_ROOT.rglob('final_report.json') if (path.parent / 'free_residual_report.json').is_file() and (path.parent / 'frozen_decoder_sweep_report.json').is_file() and (path.parent / 'unfreezing_ladder_report.json').is_file())
ARTIFACT_05D_SOURCE = next((p.resolve() for p in conditioning_candidates if p.exists()), None)
assert ARTIFACT_05D_SOURCE is not None, 'Artefatto 05d hayflow_hines_residual_conditioning non trovato.'
print({'manifest': str(COMPOSITE_MANIFEST), 'base': str(BASE_SOURCE), '05b': str(CHECKPOINT_05B_SOURCE), '05c': str(ARTIFACT_05C_SOURCE), '05d': str(ARTIFACT_05D_SOURCE)})

## 3. Composite and cryptographic provenance preflight

In [ ]:
import time
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started, hash_last = {}, {}
def hash_progress(name, done, total):
    now = time.monotonic(); hash_started.setdefault(name, now)
    percent = int(100 * done / total)
    if percent >= hash_last.get(name, -5) + 5 or done == total:
        elapsed = now - hash_started[name]; rate = done / max(elapsed, 1e-9)
        eta = (total - done) / max(rate, 1e-9)
        print(f'[HayFlow 05e][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min', flush=True)
        hash_last[name] = percent
bundle = prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST, base_source=BASE_SOURCE, progress=hash_progress)
bundle_summary = {'valid': bool(bundle.manifest['valid']), 'fingerprint': bundle.fingerprint, 'transition_count': bundle.transition_count, 'physical_merge_performed': bool(bundle.manifest['physical_merge_performed'])}
display(bundle_summary)
assert bundle_summary['valid'] and bundle_summary['transition_count'] == 29880
assert not bundle_summary['physical_merge_performed']

In [ ]:
from src.hayflow_model import HinesCapacityConfig, HinesConditioningConfig, HinesIsolationConfig, HinesPrototypeExperimentConfig, HinesSegmentCapacityExperiment
raw = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_segment_capacity.yml').read_text())
model_config = HinesPrototypeExperimentConfig.from_mapping(raw['model_experiment'])
isolation_config = HinesIsolationConfig.from_mapping(raw['isolation'])
conditioning_config = HinesConditioningConfig.from_mapping(raw['conditioning'])
capacity_config = HinesCapacityConfig.from_mapping(raw['capacity'])
OUTPUT_DIR = Path('/kaggle/working/artifacts/hayflow_hines_segment_capacity')
if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
session = HinesSegmentCapacityExperiment(bundle, OUTPUT_DIR, model_config, isolation_config, conditioning_config, capacity_config, CHECKPOINT_05B_SOURCE, ARTIFACT_05C_SOURCE, ARTIFACT_05D_SOURCE, code_revision=REVISION)
prepare_report = session.prepare_capacity_probe()
display(prepare_report)
assert prepare_report['training_contract_blockers'] == []
assert not prepare_report['full_training_authorized']

## 4. Closed-form capacity probes
Il probe condiviso misura il residuo irriducibile delle feature correnti. Il bias per segmento isola la memorizzazione statica. La famiglia low-rank usa la variazione delle feature fra i due futuri per misurare la discriminazione causale. Tutti i fit sono `numpy.linalg.lstsq`/SVD in float64: nessun optimizer iterativo e nessuna GPU per il fit.

In [ ]:
probe_report = session.run_capacity_probes()
summary = {
    'valid': probe_report['valid'],
    'shared_one_passed': probe_report['one_transition']['shared_linear']['passed'],
    'shared_pair_passed': probe_report['branch_pair']['shared_linear']['passed'],
    'segment_bias_plus_shared_pair_passed': probe_report['branch_pair']['segment_bias_plus_shared_linear']['passed'],
    'smallest_passing_segment_conditioned_rank': probe_report['smallest_passing_segment_conditioned_rank'],
    'locally_unidentifiable_segments': probe_report['branch_pair']['segment_conditioned_diagnostics']['locally_unidentifiable_segment_count'],
}
display(summary)
assert probe_report['valid']
assert not probe_report['full_training_authorized']

## 5. Inspect static memorization, branch discrimination, and rank path

In [ ]:
metric_table = pd.read_parquet(OUTPUT_DIR / 'capacity_probe_metrics.parquet')
display(metric_table[['sample', 'probe', 'rank', 'parameter_count', 'voltage_rmse_mv', 'maximum_segment_error_mv', 'branching_retention', 'static_residual_rmse_mv', 'branch_delta_residual_rmse_mv', 'passed']])
display({
    'shared_design_rank': probe_report['branch_pair']['shared_design_spectrum']['numerical_rank'],
    'shared_design_columns': probe_report['branch_pair']['shared_design_spectrum']['columns'],
    'shared_design_condition_number': probe_report['branch_pair']['shared_design_spectrum']['condition_number_nonzero'],
    'segment_coefficient_rank': probe_report['branch_pair']['segment_conditioned_diagnostics']['coefficient_matrix_rank'],
})

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUTPUT_DIR / 'capacity_diagnostics.png')))

## 6. Diagnostic verdict and artifact contract

In [ ]:
final_report = session.finalize_capacity(probe_report)
display(final_report)
assert final_report['valid']
assert not final_report['full_training_authorized']
required = ['capacity_config.json', 'capacity_probe_report.json', 'capacity_probe_metrics.parquet', 'capacity_diagnostics.png', 'final_report.json', 'artifact_index.json']
missing = [name for name in required if not (OUTPUT_DIR / name).is_file()]
assert not missing, missing
print({'diagnosis': final_report['diagnosis'], 'selected_rank': final_report['selected_segment_conditioned_rank'], 'next': final_report['next_step'], 'output_dir': str(OUTPUT_DIR)})

## 7. Browser download

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript, display
zip_base = Path('/kaggle/working/hayflow_hines_segment_capacity')
zip_path = Path(make_archive(str(zip_base), 'zip', root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name))
encoded = base64.b64encode(zip_path.read_bytes()).decode('ascii')
filename = zip_path.name
display(Javascript(f'''
const binary = atob('{encoded}');
const bytes = new Uint8Array(binary.length);
for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
const blob = new Blob([bytes], {{type: 'application/zip'}});
const url = URL.createObjectURL(blob);
const anchor = document.createElement('a');
anchor.href = url; anchor.download = '{filename}';
document.body.appendChild(anchor); anchor.click(); anchor.remove();
setTimeout(() => URL.revokeObjectURL(url), 60000);
'''))
print('Download avviato:', filename, f'({zip_path.stat().st_size / 2**20:.1f} MiB)')